# 1. Basic graph classes

The `Node` and `Graph` classes describe the graphs used in this project.
A vertex represents an individual, and an edge records a connection between
two individuals. The graphs are **simple and undirected**: a vertex cannot
connect to itself, a pair of vertices has at most one edge, and a connection
works in both directions.

Use the project's `.venv` kernel. The next notebook, `02_algorithms.ipynb`,
loads these classes automatically with `%run`. Experiments and saved results
come later in the simulation and analysis notebooks.

In [ ]:
import uuid

## A vertex: identity, neighbours, and degree

A `Node` represents one vertex. Its `neighbours` are the vertices connected
to it, and its `degree` is the number of those neighbours. The degree is
stored and updated whenever a connection changes, so algorithms can read it
without counting the neighbours again.

Each vertex also has a unique label called a UUID. Comparisons between nodes
use this label, so a vertex can still be recognised in a copy of a graph.
The labels have no role in the probability model.

`join()` and `unjoin()` change a connection for both vertices. When working
with a `Graph`, use `Graph.add_edge()` and `Graph.remove_edge()` instead: these
also update the graph's record of its edges. Invalid or repeated operations
print a short message. Use these methods rather than editing `neighbours`,
`degree`, or `id` directly, so the stored information stays consistent.

In [ ]:
class Node:
    """A vertex with a unique label and a set of neighbouring vertices.

    The stored degree is the number of neighbours. Adding or removing
    a connection updates the neighbours and degrees of both vertices.
    """

    def __init__(self):
        self.id = uuid.uuid4()
        self.neighbours = set()
        self.degree = 0

    def join(self, node):
        """Connect two distinct vertices and increase both degrees."""
        if node is self:
            print("Node cannot be joined to itself")
        elif node not in self.neighbours:
            self.neighbours.add(node)
            node.neighbours.add(self)
            self.degree += 1
            node.degree += 1
        else:
            print("Nodes are already joined")

    def unjoin(self, node):
        """Disconnect two neighbours and decrease both degrees."""
        if node in self.neighbours:
            self.neighbours.remove(node)
            node.neighbours.remove(self)
            self.degree -= 1
            node.degree -= 1
        else:
            print("Nodes are not joined")

    def __eq__(self, other):
        if not isinstance(other, Node):
            return NotImplemented
        return self.id == other.id

    def __hash__(self):
        return hash(self.id)

## A graph: vertices and unordered edges

`Graph(n)` starts with $n$ vertices and no edges. The `nodes` set holds the
vertices, and the `edges` set holds the connections between them. Each edge
is stored as a `frozenset` containing its two vertices: their order does not
matter. Removing a vertex also removes every edge attached to it.

Core pruning works on a copy so we can compare the input graph with its core.
The `__deepcopy__()` method creates separate vertices and connections while
keeping their UUID labels. Changes to either graph do not affect the other.

Generated random intersection graphs also keep a record of how they were made:

- `memberships` records how many community memberships were assigned to each
  vertex, including vertices with no memberships.
- `communities` records the vertex label assigned to each place in every
  community. These records are tuples, so they cannot be edited in place.
  One vertex can occupy more than one place in a community.

These records stay unchanged if vertices are removed or a core is taken.
Use `nodes`, `edges`, and each node's `degree` to inspect the graph as it is
now. A graph built by hand starts with empty membership records. The next
notebook explains how memberships are sampled and turned into edges.

In [ ]:
class Graph:
    """An undirected graph that also keeps its sampled membership records."""

    def __init__(self, n=0):
        self.nodes = {Node() for _ in range(n)}
        self.edges = set()
        self.memberships = {} # Only relevant in case instance is a RIG
        self.communities = () # Only relevant in case instance is a RIG

    def add_node(self, node):
        """Add a vertex without changing its label or existing connections."""
        if node not in self.nodes:
            self.nodes.add(node)
        else:
            print("Node already in graph")

    def remove_node(self, node):
        """Remove a vertex and every edge attached to it."""
        if node in self.nodes:
            # Use a copy because removing edges changes the neighbour set.
            for neighbour in node.neighbours.copy():
                self.remove_edge(node, neighbour)
            self.nodes.remove(node)
        else:
            print("Node not in graph")

    def add_edge(self, node1, node2):
        """Add an edge between two vertices already belonging to the graph."""
        if node1 not in self.nodes or node2 not in self.nodes:
            print("Nodes not in graph")
        elif node1 is node2:
            print("Node cannot be joined to itself")
        elif frozenset({node1, node2}) in self.edges:
            print("Edge already in graph")
        else:
            node1.join(node2)
            self.edges.add(frozenset({node1, node2}))

    def remove_edge(self, node1, node2):
        """Remove an edge and update both endpoints' neighbour sets."""
        edge = frozenset({node1, node2})
        if edge in self.edges:
            node1.unjoin(node2)
            self.edges.remove(edge)
        else:
            print("Edge not in graph")

    def __deepcopy__(self, memo):
        """Make an independent graph copy, keeping the same vertex labels."""
        new_graph = Graph()
        memo[id(self)] = new_graph

        # Link each input vertex to a separate copy with the same label.
        node_map = {}
        for node in self.nodes:
            new_node = Node()
            new_node.id = node.id
            node_map[node] = new_node
            memo[id(node)] = new_node
            new_graph.add_node(new_node)

        # Rebuild the edges so the two graphs have separate neighbour sets.
        for edge in self.edges:
            node1, node2 = tuple(edge)
            new_graph.add_edge(node_map[node1], node_map[node2])

        # Copy the counts; the community tuples can safely be shared.
        # Keep this sampling history even if vertices are later removed.
        new_graph.memberships = self.memberships.copy()
        new_graph.communities = self.communities
        return new_graph